In [1]:
from torchvision.models import vit_b_16, ViT_B_16_Weights
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import torch.nn.functional as F
import uuid
import json
import os
from fgsm_helperfxnsViT import (
    get_all_image_paths, get_input_batch, output_prediction, extract_true_label,
    compare_labels, fgsm_attack, save_adv_image, run_fgsm_pipeline
)

In [ ]:
root_dir = "val/val"
all_images = get_all_image_paths(root_dir)

# Constants
epsilons = [0.001, 0.01, 0.1]
correct_before = 0
total_images = 0

correct_after_per_eps = {eps: 0 for eps in epsilons} # counting correct per epsilon
total_per_eps = {eps: 0 for eps in epsilons}

# Load the model pre-trained model on ImageNet once globally
weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=weights)
model.eval()
# Preprocess and classify
preprocess = weights.transforms()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

In [ ]:
# Loop through all images
for filename in all_images:
    try:
        total_images += 1
        try:
            input_batch = get_input_batch(device, filename, preprocess)
        except Exception as e:
            print(f"💔 can't do input_batch for {filename}: {type(e).__name__}: {e}")
        # Get true label
        true_index, true_label = extract_true_label(filename)
        print(f"True label: {true_label}, index: {true_index}")

        # Get prediction before FGSM (this returns a string label)
        pred_before = output_prediction(model, input_batch)

        print(f"📊 Model predicted: {pred_before}, True index: {true_index}")

        # Compare string vs string (use true_label, not true_index)
        is_correct_before = compare_labels(pred_before, true_index) # changed this from true_label because pred_before is an index, not a string label. compare_labels should handle this correctly.
        if is_correct_before:
            correct_before += 1

        for eps in epsilons:
            # Use CORnet to generate perturbed image
            pred_after, perturbed_image = run_fgsm_pipeline(model, device, filename, eps, preprocess)
            try:
                save_adv_image(
                    perturbed_image, eps, true_label, true_index, pred_before, pred_after,
                    output_dir=f"adv_ViToutputs2/adv_ViToutputs2_eps{eps}"
                )
            except Exception as e:
                print(f"❌ Failed to save image for {filename} at eps={eps}: {e}")

            # Compare indices (both ints)
            is_correct_after = compare_labels(pred_after, true_index) 
            # Add debugging output:
            print(f"{filename} | eps={eps} | correct before? {is_correct_before} | correct after? {is_correct_after}")

            if is_correct_after:
                correct_after_per_eps[eps] += 1
        print(f"---------------------------------------------------------{filename} ends---------------------------------------------------------")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [ ]:
print(f"\nCorrect before FGSM: {correct_before}/{total_images} = {correct_before / total_images:.2%}")
for eps in epsilons:
    acc = correct_after_per_eps[eps] / total_images # total_per_eps[eps]
    print(f"Epsilon {eps}: Accuracy after FGSM = {correct_after_per_eps[eps]}/{total_images} = {acc:.2%}")
        